In [14]:
import sys
import os

# get current working directory (where your notebook or script is running)
cwd = os.getcwd()

# compute its parent directory
parent = os.path.abspath(os.path.join(cwd, os.pardir))

# insert it at the front of sys.path
if parent not in sys.path:
    sys.path.insert(0, parent)

import glob
import yaml
import cv2
import numpy as np
import torch
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm
import logging
from models import create_model, create_model2, create_im2im
import torch.nn as nn
from tqdm import tqdm
mse_loss = nn.MSELoss()
import imageio
import pickle
import matplotlib.ticker as ticker
from datasets.fastmri import FastMRIDataset
import datasets.fastmri.utils as fastmri_utils
from torch.utils.data import TensorDataset, DataLoader
from collections import defaultdict
import pandas as pd
import random
import seaborn as sns

torch.manual_seed(1)

In [15]:
from datasets.denoising_fmd.data_loader import load_noisy_single_loader

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [17]:
def load_img(img_path: str):
    img = cv2.imread(img_path, -1)
    
    img_tensor = torch.from_numpy(img).clone().detach() 
    img_tensor = img_tensor.to(torch.float32).div(255.0)
    
    return torch.clamp(img_tensor, 0, 1)   

def load_yaml(file_path: str) -> dict:
    """Loads a YAML configuration file and returns the content as a dictionary."""
    with open(file_path) as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    return config

def load_model(model_type: str, model_path: str, config_path: str, device: str):
    """
    Loads and initializes a U-Net model.

    Args:
        model_type (str): "diffusion", "quantile", "im2im".
        model_path (str): Path to the model checkpoint.
        config_path (str): Path to the YAML config file.
        device (str): Device to load the model onto ("cuda" or "cpu").

    Returns:
        tuple: (model, timevect)
    """
    # Load model configuration
    model_config = load_yaml(config_path)

    # Load and initialize the specified model
    state_dict = torch.load(model_path, map_location=device, weights_only = False)["model_state_dict"]
    
    if model_type == "diffusion":
        model = create_model(**model_config, num_measurements=1).to(device)
    elif model_type == "quantile":
        model = create_model2(**model_config, num_measurements=1).to(device)
    elif model_type == "im2im":
        model = create_im2im(**model_config, num_measurements=1).to(device)
    else:
        raise ValueError("Invalid model type. Choose 'diffusion' or 'quantile'.")

    model.load_state_dict(state_dict)

    # Define time vector
    timevect = torch.tensor([0.5], device=device)
    print("Model Loaded!")
    return model, timevect

In [ ]:
mod_type = "im2im"
path = '/share/monakhova/Cassandra_data/UQNet_proj/model_weights/poisson_im2im_58.pth'
config_path = '../models/model_config.yaml'
device = device
im2im_model, timevect = load_model(mod_type, path, config_path, device)

path = '/share/monakhova/Cassandra_data/UQNet_proj/model_weights/poisson_quantileuq_58.pth'

quant_config = '../models/model_config.yaml'
#finetuned_quantile_path = '/home/ty392/UQNet/code_for_quant/model_weights/gaussian_quantile_95.pth'
quant_model, timevect = load_model(model_type = 'quantile', model_path = path, config_path = quant_config, device = "cuda")



Model Loaded!
Model Loaded!


In [19]:
#Validation DATASET:
validation_data_path = "/share/monakhova/Cassandra_data/UQNet_proj/Denoising_FMD/validation"

noise_type = 'poisson'
min_noise = 0
max_noise = 0.5
num_noise = 25

val_dataloader = load_noisy_single_loader(noise_type, min_noise, max_noise, num_noise, image_folder=validation_data_path, batch_size=4,  types=None, transform=None)
val_dataset = val_dataloader.dataset

In [21]:
from torchmetrics import StructuralSimilarityIndexMeasure
from torchmetrics.image import PeakSignalNoiseRatio
import lpips 
dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

im2im_mse_acc = 0
im2im_ssim_acc = 0
im2im_psnr_acc = 0
im2im_lpips_acc = 0

quantile_mse_acc = 0
quantile_ssim_acc = 0
quantile_psnr_acc = 0
quantile_lpips_acc = 0

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
lpips_metric = lpips.LPIPS(net='vgg').to(device)

with torch.no_grad():
    for batch in tqdm(dataloader):
        noisy, clean = batch
        clean_im2im = clean[:, 0, :, :].to(device)
        clean = clean.to(device)

        #For Im2Im:
        curr_denoised_lam =  im2im_model(noisy.to(device))
        im2im_output = torch.unsqueeze(curr_denoised_lam[:, 1, :, :], 1)
        im2im_clean = clean.to(device)

        im2im_mse_acc += mse_loss(im2im_output, im2im_clean)
        im2im_ssim_acc += ssim_metric(im2im_output, im2im_clean)
        im2im_psnr_acc += psnr_metric(im2im_output, im2im_clean)
        im2im_lpips_acc += (torch.sum(lpips_metric(im2im_output, im2im_clean)) / im2im_output.shape[0])


        # For Quantile
        q_prediction = torch.tensor([0.5], device=device, dtype=torch.float32)
        quantile_output = quant_model(noisy.to(device), q_prediction.to(device))
        quantile_clean = clean.to(device)
        quantile_mse_acc += mse_loss(quantile_clean, quantile_output)
        quantile_ssim_acc += ssim_metric(quantile_clean, quantile_output)
        quantile_psnr_acc += psnr_metric(quantile_clean, quantile_output)
        quantile_lpips_acc += (torch.sum(lpips_metric(quantile_clean, quantile_output)) / im2im_output.shape[0])
        

print("im2im MSE: ", im2im_mse_acc.item() / len(dataloader))
print("im2im SSIM: ", im2im_ssim_acc.item() / len(dataloader))
print("im2im PSNR: ", im2im_psnr_acc.item() / len(dataloader))
print("im2im LPIPS: ", im2im_lpips_acc.item() / len(dataloader))
print("Quantile MSE: ", quantile_mse_acc.item() / len(dataloader))
print("Quantile SSIM: ", quantile_ssim_acc.item() / len(dataloader))
print("Quantile PSNR: ", quantile_psnr_acc.item() / len(dataloader))
print("Quantile LPIPS: ", quantile_lpips_acc.item() / len(dataloader))

# print(np.mean(perc_miss_im2im))
# print(np.mean(perc_miss_quant))

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/ttk22/.conda/envs/learned_uncertainty/lib/python3.8/site-packages/lpips/weights/v0.1/vgg.pth


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:27<00:00,  3.48s/it]


im2im MSE:  0.0021050875075161457
im2im SSIM:  0.8113746643066406
im2im PSNR:  26.921335220336914
im2im LPIPS:  0.34589558839797974
Quantile MSE:  0.0021241260692477226
Quantile SSIM:  0.8392511606216431
Quantile PSNR:  26.988845825195312
Quantile LPIPS:  0.32601743936538696
